Simulamos nulos y hacemos interpolación lineal

In [3]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load):
            file_path_load = alt_file_path_load
        else:
            raise FileNotFoundError("Air_Quality.csv no encontrado. Asegúrate de que esté en la carpeta 'CSVs' dentro de tu proyecto.")
    df_air_quality = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo Air_Quality.csv en la ruta esperada.")
    print("Creando DataFrame de ejemplo para continuar con la ejecución del script...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200): # Reducido para ejemplo rápido
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([
                _date, _city_ex,
                200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5,
                50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20,
                40+np.random.rand()*20
            ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
if 'CO2' in df_air_quality.columns:
    df_air_quality = df_air_quality.drop(columns=['CO2'])
    print("Columna 'CO2' descartada.")

# --- Parámetros de Simulación e Imputación ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL
N_REPETITIONS = 5

NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
INTERPOLATION_METHOD_NAME = 'interpolacionLineal'
METRICS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'metricas')
IMPUTED_CSVS_OUTPUT_DIR = os.path.join(IMPUTATIONS_BASE_DIR, INTERPOLATION_METHOD_NAME, 'CSVs')

os.makedirs(SIMULATED_DATA_ROOT_DIR, exist_ok=True)
os.makedirs(METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares ---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True)
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = []
    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']
    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0 :
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length = config['length']
        num_blocks_ratio = config.get('num_blocks_ratio', 0.01)
        num_blocks = max(1, int( (n_rows * num_blocks_ratio) / block_length if block_length > 0 else n_rows * num_blocks_ratio ))
        for _ in range(num_blocks):
            if n_rows < block_length: break
            if (n_rows - block_length) < 0: continue
            start_pos = random.randint(0, n_rows - block_length)
            current_block_positions = list(range(start_pos, start_pos + block_length))
            row_indices_to_nan_positions.extend(current_block_positions)
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))
    if row_indices_to_nan_positions:
        col_positions_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_to_nan = [pos for pos in col_positions_to_nan if pos != -1]
        if col_positions_to_nan:
             df_with_nans.iloc[row_indices_to_nan_positions, col_positions_to_nan] = np.nan
    df_original_city_reset = df_original_city_input.copy().reset_index(drop=True)
    cols_to_check_mask = [col for col in ALL_TARGET_COLS if col in df_with_nans.columns and col in df_original_city_reset.columns]
    if not cols_to_check_mask:
        introduced_nan_mask_df = pd.DataFrame(index=df_with_nans.index)
    else:
        introduced_nan_mask_df = df_with_nans[cols_to_check_mask].isnull() & ~df_original_city_reset[cols_to_check_mask].isnull()
    return df_with_nans, introduced_nan_mask_df

def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    results = {} # Usar un diccionario para almacenar todas las métricas por columna
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue

        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values

        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]

        if len(true_at_nans) == 0:
            continue
        
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]

        if len(true_at_nans_valid) == 0:
            continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            
            # Calcular rango de la variable original (sobre los valores verdaderos donde se imputó)
            # O mejor, sobre toda la columna original para tener un denominador consistente
            original_col_range = y_true_df_reset[col].max() - y_true_df_reset[col].min()
            
            nrmae_range = np.nan
            nrmse_range = np.nan
            if original_col_range > 1e-9: # Evitar división por cero o rango muy pequeño
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range

            results[col]['MAE'] = mae
            results[col]['RMSE'] = rmse
            results[col]['NRMAE_range'] = nrmae_range
            results[col]['NRMSE_range'] = nrmse_range

        except Exception as e:
            # print(f"Error calculando métricas para {col}: {e}") # Comentado
            pass # Los NaNs se mantendrán
            
    return results

# --- Script Principal ---
all_detailed_metrics_results = []
start_time_script = time.time()

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está cargado o está vacío. Carga tus datos primero.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad: {city_name_original} (Directorio seguro: {city_name_safe})")
    city_sim_data_dir_root = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe)
    os.makedirs(city_sim_data_dir_root, exist_ok=True)
    city_imputed_csv_dir_root = os.path.join(IMPUTED_CSVS_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_dir_root, exist_ok=True)

    df_city_original_for_sim = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city_original_for_sim.empty:
        print(f"  No hay datos para la ciudad {city_name_original}. Omitiendo.")
        continue
    if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
        print(f"  Advertencia: NaNs preexistentes en {city_name_original}. Rellenando con ffill/bfill para ground truth de simulación.")
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_original_for_sim.columns:
                df_city_original_for_sim[col_fill] = df_city_original_for_sim[col_fill].ffill().bfill()
        if df_city_original_for_sim[ALL_TARGET_COLS].isnull().any().any():
            print(f"  Error: No se pudieron rellenar NaNs preexistentes en {city_name_original} tras ffill/bfill. Omitiendo.")
            continue
    df_city_original_ground_truth_reset = df_city_original_for_sim.copy().reset_index(drop=True)

    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        # print(f"  Escenario de Simulación de NaNs: {scenario_name_safe}") # Comentado
        scenario_city_sim_data_dir_specific = os.path.join(city_sim_data_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_sim_data_dir_specific, exist_ok=True)
        scenario_city_imputed_csv_dir_specific = os.path.join(city_imputed_csv_dir_root, scenario_name_safe)
        os.makedirs(scenario_city_imputed_csv_dir_specific, exist_ok=True)

        for rep in range(1, N_REPETITIONS + 1):
            current_seed = int(time.time_ns() + random.randint(0,100000) + rep + hash(city_name_original) + hash(scenario_name_safe)) % (2**32 - 1)
            df_city_with_nans_reset, introduced_nan_mask_df_reset = introduce_nans_by_row(
                df_city_original_for_sim, scenario, random_seed_input=current_seed
            )
            sim_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_with_nans.csv"
            df_city_with_nans_reset.to_csv(os.path.join(scenario_city_sim_data_dir_specific, sim_filename), index=False)
            
            df_city_imputed_linear_reset = df_city_with_nans_reset.copy()
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].interpolate(
                        method='linear', limit_direction='both', limit_area=None)
            for col in ALL_TARGET_COLS:
                if col in df_city_imputed_linear_reset.columns:
                    df_city_imputed_linear_reset[col] = df_city_imputed_linear_reset[col].ffill().bfill()
            imputed_filename = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed}_imputed_linear.csv"
            df_city_imputed_linear_reset.to_csv(os.path.join(scenario_city_imputed_csv_dir_specific, imputed_filename), index=False)

            metrics_scores = calculate_metrics_with_nmae_nrmse( # Nueva función de métricas
                df_city_original_ground_truth_reset,
                df_city_imputed_linear_reset,
                introduced_nan_mask_df_reset
            )
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed, 'Imputation_Method': INTERPOLATION_METHOD_NAME,
                    'Variable': var_name, 
                    'MAE': metrics_scores[var_name]['MAE'], 
                    'RMSE': metrics_scores[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores[var_name]['NRMAE_range'], # Nueva métrica
                    'NRMSE_range': metrics_scores[var_name]['NRMSE_range']  # Nueva métrica
                })
    print(f"  Procesamiento de escenarios para {city_name_original} completado.")


# --- Guardado de Métricas Resumidas ---
if all_detailed_metrics_results:
    df_all_detailed_metrics = pd.DataFrame(all_detailed_metrics_results)
    
    # 1. metrics_variables_summary_<ciudad>.csv
    print("\nGenerando resúmenes de métricas por variable y ciudad...")
    # Ahora incluirá NRMAE_range y NRMSE_range en el promedio
    grouped_by_city_var = df_all_detailed_metrics.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var.rename(columns={
        'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 
        'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    
    for city_name_original in grouped_by_city_var['City'].unique():
        city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
        city_metrics_df = grouped_by_city_var[grouped_by_city_var['City'] == city_name_original]
        city_metrics_df_to_save = city_metrics_df[['City', 'Variable', 'Avg_MAE', 'Avg_RMSE', 'Avg_NRMAE_range', 'Avg_NRMSE_range']]
        variables_summary_path = os.path.join(METRICS_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe}.csv')
        city_metrics_df_to_save.to_csv(variables_summary_path, index=False)
        print(f"  Resumen de variables para {city_name_original} guardado en: {variables_summary_path}")
        if city_name_original == df_air_quality['City'].unique()[0]: # Imprimir solo para la primera ciudad como ejemplo
            print(city_metrics_df_to_save)


    # 2. metrics_contaminants_combined_summary.csv
    print("\nGenerando resumen combinado de contaminantes...")
    df_contaminants_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'].isin(CONTAMINANT_COLS)]
    # Promediar todas las métricas, incluyendo las normalizadas
    contaminants_combined_summary = df_contaminants_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary.rename(columns={
        'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 
        'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary.to_csv(contaminants_summary_path, index=False)
    print(f"  Resumen combinado de contaminantes guardado en: {contaminants_summary_path}")
    print(contaminants_combined_summary)

    # 3. metrics_aqi_combined_summary.csv
    print("\nGenerando resumen combinado de AQI...")
    df_aqi_only = df_all_detailed_metrics[df_all_detailed_metrics['Variable'] == 'AQI']
    # Promediar todas las métricas, incluyendo las normalizadas
    aqi_combined_summary = df_aqi_only.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary.rename(columns={
        'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI',
        'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path = os.path.join(METRICS_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary.to_csv(aqi_summary_path, index=False)
    print(f"  Resumen combinado de AQI guardado en: {aqi_summary_path}")
    print(aqi_combined_summary)
    
    # (Opcional) Guardar el ALL_metrics_detailed si aún lo quieres para inspección manual
    # global_detailed_metrics_file = os.path.join(METRICS_OUTPUT_DIR, f'ALL_metrics_detailed_{INTERPOLATION_METHOD_NAME}.csv')
    # df_all_detailed_metrics.to_csv(global_detailed_metrics_file, index=False)
    # print(f"\nTodas las métricas detalladas guardadas en: {global_detailed_metrics_file}")
else:
    print("No se generaron resultados de métricas detalladas.")

end_time_script = time.time()
print(f"\nProceso completado.")
print(f"Archivos de datos simulados guardados en subcarpetas dentro de: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos de métricas resumidas guardados en: {METRICS_OUTPUT_DIR}")
print(f"Archivos CSV imputados guardados en subcarpetas dentro de: {IMPUTED_CSVS_OUTPUT_DIR}")
print(f"Tiempo total de ejecución: {(end_time_script - start_time_script) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv
Columna 'CO2' descartada.

Procesando Ciudad: Brasilia (Directorio seguro: Brasilia)
  Procesamiento de escenarios para Brasilia completado.

Procesando Ciudad: Cairo (Directorio seguro: Cairo)
  Procesamiento de escenarios para Cairo completado.

Procesando Ciudad: Dubai (Directorio seguro: Dubai)
  Procesamiento de escenarios para Dubai completado.

Procesando Ciudad: London (Directorio seguro: London)
  Procesamiento de escenarios para London completado.

Procesando Ciudad: New York (Directorio seguro: New_York)
  Procesamiento de escenarios para New York completado.

Procesando Ciudad: Sydney (Directorio seguro: Sydney)
  Procesamiento de escenarios para Sydney completado.

Generando resúmenes de métricas por variable y ciudad...
  Resumen de variables para Brasilia guardado en: c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\imputaciones\interpolacionLineal\metricas\metrics_variables_summary_Brasilia.csv
       City Variable    Av

SARIMA ORDERS

In [ ]:
# CELDA 1: BÚSQUEDA DE ÓRDENES ÓPTIMOS ARIMA/SARIMA (EJECUTAR SOLO UNA VEZ O CUANDO SEA NECESARIO)
import pandas as pd
import numpy as np
import os
import pmdarima as pm
import json
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# --- 0. Carga de Datos (Asegúrate de que df_air_quality esté cargado) ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality = pd.read_csv(file_path_load)
    df_air_quality['Date'] = pd.to_datetime(df_air_quality['Date'])
    if 'CO2' in df_air_quality.columns: df_air_quality = df_air_quality.drop(columns=['CO2'])
    print(f"Datos cargados desde: {file_path_load} para Celda 1")
except FileNotFoundError:
    print("Error: df_air_quality no encontrado en Celda 1.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo creado para Celda 1.")


# --- Parámetros ---
COLUMNS_FOR_ARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
SEASONAL_PERIOD = 24

PROJECT_ROOT_DIR_ARIMA_SEARCH = os.getcwd()
# Directorio donde se guardarán los JSONs directamente
ORDERS_OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR_ARIMA_SEARCH, 'imputaciones', 'SARIMA_auto', 'model_orders_v2')
os.makedirs(ORDERS_OUTPUT_DIR, exist_ok=True) # Solo se crea la carpeta base 'model_orders'

print(f"--- INICIO DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")
print(f"Los órdenes se guardarán directamente en: {ORDERS_OUTPUT_DIR}")

if 'df_air_quality' not in locals() or df_air_quality.empty:
    print("Error: df_air_quality no está definido o está vacío en Celda 1.")
    exit()

for city_name_original in df_air_quality['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando ciudad: {city_name_original}")

    df_city = df_air_quality[df_air_quality['City'] == city_name_original].copy()
    if df_city.empty:
        print(f"  No hay datos para {city_name_original}, omitiendo.")
        continue

    for col_name in COLUMNS_FOR_ARIMA:
        if col_name not in df_city.columns:
            print(f"  Variable {col_name} no encontrada en los datos de {city_name_original}, omitiendo.")
            continue

        print(f"  Buscando orden para variable: {col_name}...")
        series_data = df_city[col_name].dropna()

        if len(series_data) < 2 * SEASONAL_PERIOD:
            print(f"    Datos insuficientes para SARIMA ({len(series_data)} puntos). Intentando ARIMA no estacional.")
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, d=None, seasonal=False, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)
                order = auto_model.order; seasonal_order = (0,0,0,0)
            except Exception as e:
                print(f"    Error en auto_arima no estacional para {col_name}: {e}. Usando (1,0,0).")
                order = (1,0,0); seasonal_order = (0,0,0,0)
        elif len(series_data) < 10:
             print(f"    Datos muy insuficientes (<10 puntos). Usando (1,0,0).")
             order = (1,0,0); seasonal_order = (0,0,0,0)
        else:
            try:
                auto_model = pm.auto_arima(series_data, start_p=1, start_q=1, max_p=5, max_q=5, m=SEASONAL_PERIOD, start_P=0, D=None, max_P=2, max_Q=2, seasonal=True, trace=False, error_action='ignore', suppress_warnings=True, stepwise=True)     
                order = auto_model.order; seasonal_order = auto_model.seasonal_order
                print(f"    Orden encontrado para {col_name}: {order}, Estacional: {seasonal_order}")
            except Exception as e:
                print(f"    Error en auto_arima para {col_name}: {e}. Usando (1,0,1)(1,0,1,{SEASONAL_PERIOD}).")
                order = (1,0,1); seasonal_order = (1,0,1,SEASONAL_PERIOD) if SEASONAL_PERIOD > 0 else (0,0,0,0)
        
        order_data = {'order': list(order), 'seasonal_order': list(seasonal_order)}
        # Guardar directamente en ORDERS_OUTPUT_DIR, sin subcarpeta de ciudad para el JSON
        file_path_order = os.path.join(ORDERS_OUTPUT_DIR, f'orders_{city_name_safe}_{col_name}.json')
        with open(file_path_order, 'w') as f:
            json.dump(order_data, f, indent=4)
        print(f"    Órdenes guardados en: {file_path_order}")

print(f"--- FIN DE BÚSQUEDA DE ÓRDENES ARIMA/SARIMA ---")

Datos originales (ground truth) cargados desde: CSVs\Air_Quality.csv
--- Fase 1: Encontrando/Cargando Órdenes SARIMA Óptimos ---
  Procesando ciudad para encontrar órdenes: Brasilia
    Procesando variable para auto_arima: CO
      Ejecutando auto_arima para Brasilia-CO (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-CO: Order=(3, 0, 0), Seasonal=(1, 0, 1, 24)
    Procesando variable para auto_arima: NO2
      Ejecutando auto_arima para Brasilia-NO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-NO2: Order=(1, 1, 0), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: SO2
      Ejecutando auto_arima para Brasilia-SO2 (esto puede tardar)...
        Usando subconjunto de 2160 filas para auto_arima.
      auto_arima completado para Brasilia-SO2: Order=(3, 1, 1), Seasonal=(1, 0, 0, 24)
    Procesando variable para auto_arima: O3


KeyboardInterrupt: 

Imputación SARIMA

In [2]:
# CELDA 2: IMPUTACIÓN CON SARIMA USANDO ÓRDENES GUARDADOS (CON FALLBACK Y CORRECCIÓN DE EXTREMOS)
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- Parámetros (sin cambios) ---
COLUMNS_TO_IMPUTE_SARIMA = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_REPETITIONS_SARIMA = 5
NAN_SIMULATION_SCENARIOS_SARIMA = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]
# --- Rutas de Entrada y Salida (sin cambios) ---
PROJECT_ROOT_DIR_SARIMA_IMPUTE = os.getcwd()
SIMULATED_DATA_INPUT_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'CSVs', 'simulados', 'Brasilia')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', 'SARIMA_auto', 'model_orders')
SARIMA_METHOD_NAME = 'SARIMA_auto'
IMPUTATIONS_SARIMA_BASE_DIR = os.path.join(PROJECT_ROOT_DIR_SARIMA_IMPUTE, 'imputaciones', SARIMA_METHOD_NAME)
METRICS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'metricas')
IMPUTED_CSVS_SARIMA_OUTPUT_DIR = os.path.join(IMPUTATIONS_SARIMA_BASE_DIR, 'CSVs')
os.makedirs(METRICS_SARIMA_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, exist_ok=True)

# --- Función de Imputación SARIMA MODIFICADA ---
def impute_sarima_kalman_smoother(series_with_nans, order_tuple, seasonal_order_tuple, original_series_for_bounds=None):
    imputed_series = series_with_nans.copy()
    original_nan_mask = imputed_series.isnull() # Máscara de dónde estaban los NaNs originalmente

    # Si no hay NaNs o la serie es demasiado corta, usar ffill/bfill
    # Aumentar el umbral de longitud mínima un poco más para ser conservador
    min_len_heuristic = max(sum(order_tuple), sum(seasonal_order_tuple[:3])) + seasonal_order_tuple[3] if seasonal_order_tuple[3] > 0 else max(sum(order_tuple), sum(seasonal_order_tuple[:3]))
    min_len_heuristic = max(min_len_heuristic, 10) # Mínimo absoluto

    if not original_nan_mask.any() or len(imputed_series.dropna()) < min_len_heuristic + 5 :
        # print(f"    Serie sin NaNs o demasiado corta ({len(imputed_series.dropna())} pts) para SARIMA. Usando ffill/bfill.")
        return series_with_nans.ffill().bfill() # Usar series_with_nans para ffill/bfill

    sarima_failed_or_unstable = False
    try:
        model = SARIMAX(imputed_series, # Usar la serie con NaNs
                        order=order_tuple,
                        seasonal_order=seasonal_order_tuple,
                        enforce_stationarity=False, # Permitir modelos no estacionarios
                        enforce_invertibility=False,# Permitir modelos no invertibles
                        initialization='approximate_diffuse',
                        # simple_differencing=True # Podría ayudar si hay problemas de convergencia con D > 0
                        ) 
        results = model.fit(disp=False, maxiter=200) # Aumentar maxiter
        
        # Usar .apply para obtener el estado suavizado que representa los valores imputados
        # O .predict() como antes
        predictions_all = results.predict(start=imputed_series.index[0], end=imputed_series.index[-1])
        
        # Actualizar solo donde había NaNs
        imputed_series.loc[original_nan_mask] = predictions_all[original_nan_mask]

        # Si después de la imputación SARIMA aún quedan NaNs (poco probable con predict, pero posible con otros métodos)
        if imputed_series.isnull().any():
            # print(f"    SARIMA dejó {imputed_series.isnull().sum()} NaNs. Rellenando esos con lineal sobre 'series_with_nans'.")
            temp_linear_fill = series_with_nans.interpolate(method='linear', limit_direction='both')
            imputed_series = imputed_series.fillna(temp_linear_fill) # Rellenar los NaNs restantes en imputed_series
            if imputed_series.isnull().any(): # Si lineal también falla (extremos de series_with_nans)
                imputed_series = imputed_series.ffill().bfill()

    except Exception as e:
        print(f"    Error crítico durante el ajuste/predicción SARIMA: {e}. Usando fallback completo a lineal sobre 'series_with_nans'.")
        sarima_failed_or_unstable = True

    if sarima_failed_or_unstable:
        imputed_series = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
    
    # Post-procesamiento para valores extremos en los puntos que fueron originalmente NaN
    if original_series_for_bounds is not None and original_nan_mask.any():
        # Usar la serie original (limpia) para definir límites robustos
        clean_original_data = original_series_for_bounds.dropna()
        if not clean_original_data.empty:
            q01 = clean_original_data.quantile(0.001)
            q99 = clean_original_data.quantile(0.999)
            iqr_val = clean_original_data.quantile(0.75) - clean_original_data.quantile(0.25)
            
            # Límites más generosos para evitar cortar demasiado
            # Multiplicador de IQR más grande o un buffer absoluto.
            # Evitar que los límites sean idénticos si la serie es muy constante.
            padding = max(1.0, 0.1 * abs(q99 - q01)) if (q99 - q01) != 0 else 1.0
            
            lower_bound = q01 - 3 * iqr_val - padding 
            upper_bound = q99 + 3 * iqr_val + padding
            
            # Asegurar que los límites no sean más restrictivos que min/max si los datos no tienen muchos outliers
            min_orig = clean_original_data.min()
            max_orig = clean_original_data.max()
            lower_bound = min(lower_bound, min_orig - padding)
            upper_bound = max(upper_bound, max_orig + padding)


            # Identificar valores imputados extremos SOLO donde originalmente había NaNs
            # y donde la serie original (bounds) no era NaN
            points_to_check_for_extremes = original_nan_mask & ~original_series_for_bounds.isnull()
            
            if points_to_check_for_extremes.any():
                extreme_values_mask = points_to_check_for_extremes & \
                                      ((imputed_series < lower_bound) | (imputed_series > upper_bound))
                
                if extreme_values_mask.any():
                    num_extreme = extreme_values_mask.sum()
                    print(f"    Detectados {num_extreme} valores SARIMA extremos (fuera de [{lower_bound:.2f}, {upper_bound:.2f}]). Re-imputando con lineal.")
                    
                    # Fallback a lineal para esos puntos específicos, usando 'series_with_nans' como base para la interpolación
                    temp_linear_imputation_for_extremes = series_with_nans.interpolate(method='linear', limit_direction='both').ffill().bfill()
                    imputed_series.loc[extreme_values_mask] = temp_linear_imputation_for_extremes[extreme_values_mask]
    
    return imputed_series

# --- Función para calcular métricas (sin cambios lógicos) ---
def calculate_metrics_sarima(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    # ... (código de la función sin cambios) ...
    mae_vals, rmse_vals = {}, {}
    for col in COLUMNS_TO_IMPUTE_SARIMA:
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        true_col_vals = y_true_df_reset[col].values; pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values
        true_at_nans = true_col_vals[mask_col_flags]; pred_at_nans = pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]; pred_at_nans_valid = pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: mae_vals[col], rmse_vals[col] = np.nan, np.nan; continue
        try:
            mae_vals[col] = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse_vals[col] = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
        except Exception: mae_vals[col], rmse_vals[col] = np.nan, np.nan
    return mae_vals, rmse_vals

# --- Script Principal de Imputación SARIMA (con llamada a función modificada) ---
all_detailed_metrics_sarima_results = []
start_time_script_sarima = time.time()
try:
    file_path_load_orig = os.path.join('CSVs', 'Air_Quality.csv')
    # ... (lógica de carga completa) ...
    if not os.path.exists(file_path_load_orig):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load_orig = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load_orig): file_path_load_orig = alt_file_path_load_orig
        else: raise FileNotFoundError("Air_Quality.csv (original) no encontrado.")
    df_air_quality_original_gt = pd.read_csv(file_path_load_orig)
    df_air_quality_original_gt['Date'] = pd.to_datetime(df_air_quality_original_gt['Date'])
    if 'CO2' in df_air_quality_original_gt.columns: df_air_quality_original_gt = df_air_quality_original_gt.drop(columns=['CO2'])
    print(f"Datos originales GT cargados desde: {file_path_load_orig} para Celda 2")
except FileNotFoundError:
    print("Error: df_air_quality_original_gt no encontrado en Celda 2.")
    _example_data = [] # Datos de ejemplo mínimos
    _cities_example = ['Brasilia', 'Cairo']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(24*7*4): # 4 semanas de datos horarios
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 40+np.random.rand()*20 ])
    df_air_quality_original_gt = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'AQI'])
    print("DataFrame de ejemplo GT creado para Celda 2.")


if 'df_air_quality_original_gt' not in locals() or df_air_quality_original_gt.empty:
    print("Error: df_air_quality_original_gt no está definido o está vacío en Celda 2.")
    exit()

for city_name_original_sarima in df_air_quality_original_gt['City'].unique():
    city_name_safe_sarima = city_name_original_sarima.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación SARIMA: {city_name_original_sarima}")
    # ... (definición de rutas sin cambios) ...
    ORDERS_JSON_INPUT_DIR_CITY = os.path.join(ORDERS_JSON_INPUT_DIR, city_name_safe_sarima) # Para la estructura que SÍ crea subcarpetas de ciudad para los JSON
                                                                                        # Si no hay subcarpetas, usar ORDERS_JSON_INPUT_DIR directamente.
                                                                                        # Basado en tu error anterior, asumiré que los JSONs están en ORDERS_JSON_INPUT_DIR directamente.

    # city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR, city_name_safe_sarima)
    city_sim_data_input_dir_root = os.path.join(SIMULATED_DATA_INPUT_ROOT_DIR)
    city_metrics_sarima_file_path = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'metrics_{SARIMA_METHOD_NAME}_{city_name_safe_sarima}.csv')
    city_imputed_csv_sarima_dir_root = os.path.join(IMPUTED_CSVS_SARIMA_OUTPUT_DIR, city_name_safe_sarima)
    os.makedirs(city_imputed_csv_sarima_dir_root, exist_ok=True)

    df_city_gt_for_metrics = df_air_quality_original_gt[df_air_quality_original_gt['City'] == city_name_original_sarima].copy()
    if df_city_gt_for_metrics[COLUMNS_TO_IMPUTE_SARIMA].isnull().any().any():
        for col_fill_gt in COLUMNS_TO_IMPUTE_SARIMA:
            if col_fill_gt in df_city_gt_for_metrics.columns:
                df_city_gt_for_metrics[col_fill_gt] = df_city_gt_for_metrics[col_fill_gt].ffill().bfill()
    df_city_gt_for_metrics_reset = df_city_gt_for_metrics.reset_index(drop=True)
    city_scenario_summary_metrics_sarima = []

    for scenario_sarima in NAN_SIMULATION_SCENARIOS_SARIMA:
        scenario_name_safe_sarima = scenario_sarima['name'].replace("%", "percent").replace(" ", "_")
        print(f"  Escenario: {scenario_name_safe_sarima}")
        scenario_city_sim_data_input_dir = os.path.join(city_sim_data_input_dir_root, scenario_name_safe_sarima)
        scenario_city_imputed_csv_sarima_dir = os.path.join(city_imputed_csv_sarima_dir_root, scenario_name_safe_sarima)
        os.makedirs(scenario_city_imputed_csv_sarima_dir, exist_ok=True)
        scenario_rep_maes_s = {var: [] for var in COLUMNS_TO_IMPUTE_SARIMA}
        scenario_rep_rmses_s = {var: [] for var in COLUMNS_TO_IMPUTE_SARIMA}

        for rep_s in range(1, N_REPETITIONS_SARIMA + 1):
            # ... (lógica para encontrar sim_file_path_to_load y current_seed_loaded) ...
            sim_files_in_dir = sorted([f for f in os.listdir(scenario_city_sim_data_input_dir) if f.startswith(f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}") and f.endswith("_with_nans.csv")])
            if not sim_files_in_dir:
                print(f"    Advertencia: No se encontró archivo simulado para {city_name_safe_sarima}, {scenario_name_safe_sarima}, rep {rep_s}. Omitiendo.")
                continue
            sim_file_path_to_load = os.path.join(scenario_city_sim_data_input_dir, sim_files_in_dir[0])
            current_seed_loaded = sim_files_in_dir[0].split('_seed')[-1].split('_with_nans.csv')[0]
            try:
                df_city_with_nans_loaded = pd.read_csv(sim_file_path_to_load)
            except FileNotFoundError:
                print(f"    Error: Archivo simulado no encontrado: {sim_file_path_to_load}"); continue
            
            df_city_imputed_sarima = df_city_with_nans_loaded.copy()
            mask_for_metrics_s = pd.DataFrame(False, index=df_city_gt_for_metrics_reset.index, columns=COLUMNS_TO_IMPUTE_SARIMA)
            for col_mask in COLUMNS_TO_IMPUTE_SARIMA:
                 if col_mask in df_city_with_nans_loaded.columns and col_mask in df_city_gt_for_metrics_reset.columns:
                    mask_for_metrics_s[col_mask] = df_city_with_nans_loaded[col_mask].isnull() & ~df_city_gt_for_metrics_reset[col_mask].isnull()

            for col_impute_s in COLUMNS_TO_IMPUTE_SARIMA:
                # Asumiendo que los JSONs están directamente en ORDERS_JSON_INPUT_DIR (sin subcarpeta de ciudad para el JSON)
                order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe_sarima}_{col_impute_s}.json')
                try:
                    with open(order_file_path, 'r') as f: orders = json.load(f)
                    order_tuple = tuple(orders['order']); seasonal_order_tuple = tuple(orders['seasonal_order'])
                except FileNotFoundError:
                    print(f"      Archivo de órdenes no encontrado: {order_file_path}. Usando (1,0,1)(0,0,0,0).")
                    order_tuple = (1,0,1); seasonal_order_tuple = (0,0,0,0) # Default
                except Exception as e_json: # Capturar otros errores de JSON
                    print(f"      Error cargando/parseando JSON {order_file_path}: {e_json}. Usando (1,0,1)(0,0,0,0).")
                    order_tuple = (1,0,1); seasonal_order_tuple = (0,0,0,0)

                series_to_impute = df_city_with_nans_loaded[col_impute_s].copy()
                original_clean_series_for_bounds = df_city_gt_for_metrics_reset[col_impute_s].copy() # Ground truth para límites

                imputed_values_col = impute_sarima_kalman_smoother(
                    series_to_impute,
                    order_tuple,
                    seasonal_order_tuple,
                    original_series_for_bounds=original_clean_series_for_bounds # Pasar serie limpia para bounds
                )
                df_city_imputed_sarima[col_impute_s] = imputed_values_col
            
            # ... (guardado de CSV imputado y cálculo/almacenamiento de métricas sin cambios) ...
            imputed_sarima_filename = f"{city_name_safe_sarima}_{scenario_name_safe_sarima}_rep{rep_s}_seed{current_seed_loaded}_imputed_sarima.csv"
            df_city_imputed_sarima.to_csv(os.path.join(scenario_city_imputed_csv_sarima_dir, imputed_sarima_filename), index=False)
            mae_s, rmse_s = calculate_metrics_sarima(df_city_gt_for_metrics_reset, df_city_imputed_sarima, mask_for_metrics_s)
            for var_n in COLUMNS_TO_IMPUTE_SARIMA:
                all_detailed_metrics_sarima_results.append({'City': city_name_original_sarima, 'Scenario': scenario_name_safe_sarima, 'Repetition': rep_s, 'Seed': current_seed_loaded, 'Imputation_Method': SARIMA_METHOD_NAME, 'Variable': var_n, 'MAE': mae_s.get(var_n, np.nan), 'RMSE': rmse_s.get(var_n, np.nan)})
                if not np.isnan(mae_s.get(var_n, np.nan)): scenario_rep_maes_s[var_n].append(mae_s.get(var_n))
                if not np.isnan(rmse_s.get(var_n, np.nan)): scenario_rep_rmses_s[var_n].append(rmse_s.get(var_n))
        avg_mae_aqi_s = np.nanmean(scenario_rep_maes_s.get('AQI', [np.nan])); avg_rmse_aqi_s = np.nanmean(scenario_rep_rmses_s.get('AQI', [np.nan]))
        print(f"    Promedio SARIMA para Escenario '{scenario_name_safe_sarima}' (AQI): MAE={avg_mae_aqi_s:.4f}, RMSE={avg_rmse_aqi_s:.4f}")
        for var_n_s in COLUMNS_TO_IMPUTE_SARIMA: city_scenario_summary_metrics_sarima.append({'Scenario': scenario_name_safe_sarima, 'Variable': var_n_s, 'Avg_MAE': np.nanmean(scenario_rep_maes_s.get(var_n_s, [np.nan])), 'Avg_RMSE': np.nanmean(scenario_rep_rmses_s.get(var_n_s, [np.nan])), 'Std_MAE': np.nanstd(scenario_rep_maes_s.get(var_n_s, [np.nan])), 'Std_RMSE': np.nanstd(scenario_rep_rmses_s.get(var_n_s, [np.nan]))})
    if city_scenario_summary_metrics_sarima: pd.DataFrame(city_scenario_summary_metrics_sarima).to_csv(city_metrics_sarima_file_path, index=False); print(f"  Resumen SARIMA para {city_name_original_sarima} guardado: {city_metrics_sarima_file_path}")
if all_detailed_metrics_sarima_results:
    df_all_detailed_s = pd.DataFrame(all_detailed_metrics_sarima_results)
    global_detailed_s_file = os.path.join(METRICS_SARIMA_OUTPUT_DIR, f'ALL_metrics_detailed_{SARIMA_METHOD_NAME}.csv')
    df_all_detailed_s.to_csv(global_detailed_s_file, index=False)
    print(f"\nMétricas detalladas SARIMA: {global_detailed_s_file}");
    if not df_all_detailed_s.empty: print(df_all_detailed_s.head())
end_time_script_sarima = time.time()
print(f"\nProceso SARIMA completado. CSVs: {IMPUTED_CSVS_SARIMA_OUTPUT_DIR}, Métricas: {METRICS_SARIMA_OUTPUT_DIR}")
print(f"Tiempo SARIMA: {(end_time_script_sarima - start_time_script_sarima) / 60:.2f} min.")

Datos originales GT cargados desde: CSVs\Air_Quality.csv para Celda 2

Procesando Ciudad para Imputación SARIMA: Brasilia
  Escenario: RandomRows_10percent
    Detectados 2 valores SARIMA extremos (fuera de [-4.50, 9.52]). Re-imputando con lineal.
    Detectados 3 valores SARIMA extremos (fuera de [-4.50, 9.52]). Re-imputando con lineal.
    Detectados 4 valores SARIMA extremos (fuera de [-4.50, 9.52]). Re-imputando con lineal.
    Detectados 1 valores SARIMA extremos (fuera de [-4.50, 9.52]). Re-imputando con lineal.


KeyboardInterrupt: 

RANDOM FOREST

In [ ]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler # Para escalar features
from sklearn.pipeline import Pipeline # Para combinar scaler e imputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
import time
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Carga de Datos y Preparación Inicial ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    if not os.path.exists(file_path_load):
        script_dir_parent = os.path.dirname(os.getcwd())
        alt_file_path_load = os.path.join(script_dir_parent, 'CSVs', 'Air_Quality.csv')
        if os.path.exists(alt_file_path_load): file_path_load = alt_file_path_load
        else: raise FileNotFoundError("Air_Quality.csv no encontrado.")
    df_air_quality_orig_load = pd.read_csv(file_path_load)
    print(f"Datos cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado. Creando DataFrame de ejemplo...")
    _example_data = []
    _cities_example = ['Brasilia', 'Cairo', 'Dubai', 'London', 'New_York', 'Sydney']
    _start_date = pd.to_datetime('2024-01-01')
    for _city_ex in _cities_example:
        for i in range(200):
            _date = _start_date + pd.Timedelta(hours=i)
            _example_data.append([_date, _city_ex, 200+np.random.rand()*100, 20+np.random.rand()*10, 5+np.random.rand()*5, 50+np.random.rand()*30, 15+np.random.rand()*10, 30+np.random.rand()*20, 40+np.random.rand()*20])
    df_air_quality_orig_load = pd.DataFrame(_example_data, columns=['Date', 'City', 'CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI'])
    print("DataFrame de ejemplo creado.")

df_air_quality_orig_load['Date'] = pd.to_datetime(df_air_quality_orig_load['Date'])
if 'CO2' in df_air_quality_orig_load.columns:
    df_air_quality_orig_load = df_air_quality_orig_load.drop(columns=['CO2'])

# --- Parámetros ---
CONTAMINANT_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
AQI_COL = ['AQI']
ALL_TARGET_COLS = CONTAMINANT_COLS + AQI_COL # Columnas que originalmente tendrán NaNs y serán evaluadas

# Features de tiempo
df_air_quality_orig_load['Hour'] = df_air_quality_orig_load['Date'].dt.hour
df_air_quality_orig_load['DayOfWeek'] = df_air_quality_orig_load['Date'].dt.dayofweek
df_air_quality_orig_load['Month'] = df_air_quality_orig_load['Date'].dt.month
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# Creación de Lags (ANTES de la simulación de NaNs en el bucle principal)
N_LAGS = [1, 24] # Lags a crear para cada variable en ALL_TARGET_COLS
LAG_FEATURES = []
df_with_features = df_air_quality_orig_load.copy()
for col_to_lag in ALL_TARGET_COLS:
    for lag_val in N_LAGS:
        lag_col_name = f"{col_to_lag}_lag{lag_val}"
        df_with_features[lag_col_name] = df_with_features.groupby('City')[col_to_lag].shift(lag_val)
        LAG_FEATURES.append(lag_col_name)

# Columnas que el imputer usará como features. Incluye originales, de tiempo y lags.
# El imputer intentará imputar NaNs en CUALQUIERA de estas si los encuentra.
COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM = ALL_TARGET_COLS + TIME_FEATURES + LAG_FEATURES

N_REPETITIONS = 1
NAN_SIMULATION_SCENARIOS = [
    {'name': 'RandomRows_10percent', 'type': 'row_percentage', 'config': {'percentage': 0.10}},
    {'name': 'RandomRows_20percent', 'type': 'row_percentage', 'config': {'percentage': 0.20}},
    {'name': 'BlockRows_2h', 'type': 'block_rows', 'config': {'length': 2, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_4h', 'type': 'block_rows', 'config': {'length': 4, 'num_blocks_ratio': 0.05}},
    {'name': 'BlockRows_8h', 'type': 'block_rows', 'config': {'length': 8, 'num_blocks_ratio': 0.025}},
    {'name': 'BlockRows_12h', 'type': 'block_rows', 'config': {'length': 12, 'num_blocks_ratio': 0.015}},
    {'name': 'BlockRows_24h', 'type': 'block_rows', 'config': {'length': 24, 'num_blocks_ratio': 0.01}},
]

PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados') # Leer desde aquí

RF_IMPUTER_METHOD_NAME = 'RandomForestImputer_withLags' # Nombre descriptivo
IMPUTATIONS_RF_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', RF_IMPUTER_METHOD_NAME)
METRICS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'metricas')
IMPUTED_CSVS_RF_OUTPUT_DIR = os.path.join(IMPUTATIONS_RF_BASE_DIR, 'CSVs')

os.makedirs(METRICS_RF_OUTPUT_DIR, exist_ok=True)
os.makedirs(IMPUTED_CSVS_RF_OUTPUT_DIR, exist_ok=True)

# --- Funciones Auxiliares ---
def introduce_nans_by_row(df_original_city_input, scenario_config_input, random_seed_input):
    # df_original_city_input ya viene con lags y features de tiempo
    np.random.seed(random_seed_input)
    random.seed(random_seed_input)
    df_with_nans = df_original_city_input.copy().reset_index(drop=True) # Trabajar con copia e índice reseteado
    n_rows = len(df_with_nans)
    row_indices_to_nan_positions = [] # Posiciones (0 a n-1)

    scenario_type = scenario_config_input['type']
    config = scenario_config_input['config']

    if scenario_type == 'row_percentage':
        percentage = config['percentage']
        num_rows_to_nan = int(n_rows * percentage)
        if num_rows_to_nan > 0 and n_rows > 0:
            row_indices_to_nan_positions = random.sample(range(n_rows), min(num_rows_to_nan, n_rows))
    elif scenario_type == 'block_rows':
        block_length = config['length']
        num_blocks_ratio = config.get('num_blocks_ratio', 0.01)
        num_blocks = max(1, int((n_rows * num_blocks_ratio) / block_length if block_length > 0 else n_rows * num_blocks_ratio))
        for _ in range(num_blocks):
            if n_rows < block_length or (n_rows - block_length) < 0: break
            start_pos = random.randint(0, n_rows - block_length)
            row_indices_to_nan_positions.extend(list(range(start_pos, start_pos + block_length)))
        row_indices_to_nan_positions = sorted(list(set(row_indices_to_nan_positions)))

    if row_indices_to_nan_positions:
        # Poner NaN SOLO en las columnas objetivo originales (ALL_TARGET_COLS)
        # Las columnas de lag que se vuelvan NaN como consecuencia de esto está bien.
        # Las features de tiempo NO deben tener NaNs introducidos.
        col_positions_target_to_nan = df_with_nans.columns.get_indexer(ALL_TARGET_COLS)
        col_positions_target_to_nan = [pos for pos in col_positions_target_to_nan if pos != -1]
        if col_positions_target_to_nan:
            df_with_nans.iloc[row_indices_to_nan_positions, col_positions_target_to_nan] = np.nan
    
    # Máscara de NaNs introducidos (solo para ALL_TARGET_COLS)
    df_original_city_reset = df_original_city_input.copy().reset_index(drop=True)
    introduced_nan_mask_df = pd.DataFrame(False, index=df_with_nans.index, columns=ALL_TARGET_COLS)
    for col_mask in ALL_TARGET_COLS:
        if col_mask in df_with_nans.columns and col_mask in df_original_city_reset.columns:
             introduced_nan_mask_df[col_mask] = df_with_nans[col_mask].isnull() & ~df_original_city_reset[col_mask].isnull()
    
    return df_with_nans, introduced_nan_mask_df


def calculate_metrics_with_nmae_nrmse(y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset):
    # y_true_df_reset, y_pred_df_reset, nan_mask_for_metrics_df_reset solo contienen ALL_TARGET_COLS
    results = {}
    for col in ALL_TARGET_COLS:
        results[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        if col not in y_true_df_reset.columns or col not in y_pred_df_reset.columns or col not in nan_mask_for_metrics_df_reset.columns:
            continue
        true_col_vals = y_true_df_reset[col].values
        pred_col_vals = y_pred_df_reset[col].values
        mask_col_flags = nan_mask_for_metrics_df_reset[col].values
        true_at_nans = true_col_vals[mask_col_flags]
        pred_at_nans = pred_col_vals[mask_col_flags]
        if len(true_at_nans) == 0: continue
        valid_pred_mask = ~np.isnan(pred_at_nans)
        true_at_nans_valid = true_at_nans[valid_pred_mask]
        pred_at_nans_valid = pred_at_nans[valid_pred_mask]
        if len(true_at_nans_valid) == 0: continue
        try:
            mae = mean_absolute_error(true_at_nans_valid, pred_at_nans_valid)
            rmse = np.sqrt(mean_squared_error(true_at_nans_valid, pred_at_nans_valid))
            # Usar el rango del ground truth completo de la columna para la normalización
            original_col_full_series = y_true_df_reset[col].dropna()
            original_col_range = original_col_full_series.max() - original_col_full_series.min() if not original_col_full_series.empty else 0

            nrmae_range, nrmse_range = np.nan, np.nan
            if original_col_range > 1e-9:
                nrmae_range = mae / original_col_range
                nrmse_range = rmse / original_col_range
            results[col]['MAE'], results[col]['RMSE'] = mae, rmse
            results[col]['NRMAE_range'], results[col]['NRMSE_range'] = nrmae_range, nrmse_range
        except Exception: pass
    return results

# --- Script Principal ---
all_detailed_metrics_rf_results = []
script_start_time_rf = time.time()

if 'df_with_features' not in locals() or df_with_features.empty:
    print("Error: df_with_features (datos con lags y tiempo) no está definido o está vacío.")
    exit()

for city_name_original in df_with_features['City'].unique():
    city_name_safe = city_name_original.replace(" ", "_").replace("-", "_")
    print(f"\nProcesando Ciudad para Imputación RF: {city_name_original}")

    # Datos originales (con lags y features de tiempo) para esta ciudad, usados como base para simular NaNs
    df_city_original_features = df_with_features[df_with_features['City'] == city_name_original].copy()
    if df_city_original_features.empty: continue

    # Ground truth (sin lags, solo ALL_TARGET_COLS) para calcular métricas, pre-rellenado
    df_city_gt_eval = df_air_quality_orig_load[df_air_quality_orig_load['City'] == city_name_original].copy()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any():
        for col_fill in ALL_TARGET_COLS:
             if col_fill in df_city_gt_eval.columns:
                df_city_gt_eval[col_fill] = df_city_gt_eval[col_fill].ffill().bfill()
    if df_city_gt_eval[ALL_TARGET_COLS].isnull().any().any(): # Si aún hay NaNs, omitir
        print(f"  Error fatal: No se pudieron rellenar NaNs en el ground truth de {city_name_original}. Omitiendo.")
        continue
    df_city_original_ground_truth_reset = df_city_gt_eval.reset_index(drop=True) # Solo columnas originales + tiempo


    city_imputed_csv_rf_dir_root = os.path.join(IMPUTED_CSVS_RF_OUTPUT_DIR, city_name_safe)
    os.makedirs(city_imputed_csv_rf_dir_root, exist_ok=True)

    # Definir el Imputer y el Scaler
    # --- AJUSTES EN PARÁMETROS DEL IMPUTER (VERSIÓN RÁPIDA PARA PRUEBA) ---
    rf_estimator_test = RandomForestRegressor(
        n_estimators=10,       # MUY BAJO, solo para prueba de velocidad
        random_state=42,
        n_jobs=-1,             # Usar todos los cores
        max_depth=8,           # Reducido drásticamente
        min_samples_split=10,  # Razonable
        min_samples_leaf=5,    # Razonable
        # max_features='sqrt'  # Puedes mantenerlo o quitarlo para prueba
    )
    iterative_imputer_rf = IterativeImputer(
        estimator=rf_estimator_test,
        max_iter=3,              # MUY BAJO, solo para ver si converge algo
        tol=1e-2,                # Tolerancia menos estricta
        random_state=42,
        initial_strategy='mean', # 'mean' es a menudo más rápido que 'median' para el inicio
        # imputation_order='ascending', # El default, puede ser más rápido que 'random'
        verbose=1                # PONER verbose=1 PARA VER QUÉ ESTÁ HACIENDO
    )
    scaler = StandardScaler() # Usaremos scaler antes del imputer

    # Ajustar el Scaler y el Imputer usando los datos ORIGINALES COMPLETOS de la ciudad
    # (con features de tiempo y lags, pero SIN los NaNs simulados artificialmente aún)
    # Los NaNs de los lags (al principio de la serie) serán manejados por el imputer.
    data_for_fitting_imputer = df_city_original_features[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
    
    print(f"  Ajustando Scaler e IterativeImputer con RandomForest para {city_name_original}...")
    start_fit_time = time.time()
    try:
        # 1. Escalar los datos
        scaled_data_for_fitting = scaler.fit_transform(data_for_fitting_imputer)
        # 2. Ajustar el imputer con los datos escalados
        iterative_imputer_rf.fit(scaled_data_for_fitting)
        fit_duration = time.time() - start_fit_time
        print(f"  Scaler e Imputer ajustados para {city_name_original} en {fit_duration:.2f} segundos.")
    except Exception as e_fit:
        print(f"  Error ajustando Scaler/IterativeImputer para {city_name_original}: {e_fit}. Omitiendo ciudad.")
        continue


    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = scenario['name'].replace("%", "percent").replace(" ", "_")
        print(f"    Escenario: {scenario_name_safe}")
        
        # Los datos simulados con NaNs se LEEN desde las carpetas generadas por el script anterior
        # No se vuelven a generar aquí.
        sim_data_scenario_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario_name_safe)
        imputed_rf_scenario_dir = os.path.join(city_imputed_csv_rf_dir_root, scenario_name_safe)
        os.makedirs(imputed_rf_scenario_dir, exist_ok=True)

        if not os.path.exists(sim_data_scenario_dir):
            print(f"      Directorio de datos simulados no encontrado: {sim_data_scenario_dir}. Omitiendo escenario.")
            continue

        for rep in range(1, N_REPETITIONS + 1):
            matching_files = [f for f in os.listdir(sim_data_scenario_dir) if f.startswith(f"{city_name_safe}_{scenario_name_safe}_rep{rep}") and f.endswith("_with_nans.csv")]
            if not matching_files: continue
            sim_file_path_to_load = os.path.join(sim_data_scenario_dir, matching_files[0])
            current_seed_loaded = matching_files[0].split('_seed')[-1].split('_with_nans.csv')[0]

            try:
                # df_city_with_nans_loaded_raw contendrá Date, City, y ALL_TARGET_COLS con NaNs simulados
                df_city_with_nans_loaded_raw = pd.read_csv(sim_file_path_to_load)
                
                # Reconstruir df_city_with_nans_features para el imputer:
                # Empezar con el df original de la ciudad (que tiene lags y time features)
                df_for_imputation_base = df_city_original_features.copy().reset_index(drop=True)
                
                # Sobrescribir las columnas ALL_TARGET_COLS con los valores (y NaNs) del archivo simulado
                # Asegurarse de que los índices estén alineados (ambos reseteados)
                df_city_with_nans_loaded_reset = df_city_with_nans_loaded_raw.reset_index(drop=True)
                for col_target in ALL_TARGET_COLS:
                    if col_target in df_city_with_nans_loaded_reset.columns:
                        df_for_imputation_base[col_target] = df_city_with_nans_loaded_reset[col_target]
                
                # Ahora df_for_imputation_base tiene:
                # - Lags (con NaNs al principio y donde los originales se volvieron NaN)
                # - Time features (completas)
                # - ALL_TARGET_COLS (con NaNs simulados)

            except Exception as e_load:
                print(f"      Error cargando/procesando {sim_file_path_to_load}: {e_load}.")
                continue
            
            # La máscara de NaNs introducidos se refiere a las columnas ALL_TARGET_COLS
            # Se basa en la comparación con el ground truth original SIN lags.
            temp_introduced_nan_mask_df = pd.DataFrame(False, index=df_city_original_ground_truth_reset.index, columns=ALL_TARGET_COLS)
            for col_mask_check in ALL_TARGET_COLS:
                 if col_mask_check in df_city_with_nans_loaded_reset.columns and col_mask_check in df_city_original_ground_truth_reset.columns:
                    temp_introduced_nan_mask_df[col_mask_check] = df_city_with_nans_loaded_reset[col_mask_check].isnull() & \
                                                                ~df_city_original_ground_truth_reset[col_mask_check].isnull()

            # Preparar datos para el imputer (solo columnas relevantes)
            df_to_impute_for_rf = df_for_imputation_base[COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM].copy()
            
            try:
                start_transform_time = time.time()
                # 1. Escalar los datos con NaNs
                scaled_data_to_impute = scaler.transform(df_to_impute_for_rf)
                # 2. Imputar los datos escalados
                imputed_scaled_data_rf_array = iterative_imputer_rf.transform(scaled_data_to_impute)
                # 3. Desescalar los datos imputados
                imputed_data_rf_array = scaler.inverse_transform(imputed_scaled_data_rf_array)
                transform_duration = time.time() - start_transform_time
                
                # Reconstruir el DataFrame imputado
                df_imputed_features = pd.DataFrame(imputed_data_rf_array, columns=COLUMNS_FOR_IMPUTER_TRAINING_AND_TRANSFORM)
                
                # Crear el DataFrame final para guardar (Date, City, ALL_TARGET_COLS imputados, y TIME_FEATURES)
                df_final_imputed_rf = df_for_imputation_base[['Date', 'City']].copy() # Mantener Date y City originales
                for col_final in ALL_TARGET_COLS + TIME_FEATURES: # Guardar también time features
                    if col_final in df_imputed_features.columns:
                         df_final_imputed_rf[col_final] = df_imputed_features[col_final]

            except Exception as e_impute:
                print(f"      Error durante la transformación/imputación RF para rep {rep}: {e_impute}. Omitiendo.")
                continue

            imputed_filename_rf = f"{city_name_safe}_{scenario_name_safe}_rep{rep}_seed{current_seed_loaded}_imputed_rf.csv"
            df_final_imputed_rf.to_csv(os.path.join(imputed_rf_scenario_dir, imputed_filename_rf), index=False)

            metrics_scores_rf = calculate_metrics_with_nmae_nrmse(
                df_city_original_ground_truth_reset[ALL_TARGET_COLS], # Ground truth (solo cols objetivo)
                df_final_imputed_rf[ALL_TARGET_COLS],                 # Imputados (solo cols objetivo)
                temp_introduced_nan_mask_df                           # Máscara (solo cols objetivo)
            )
            for var_name in ALL_TARGET_COLS:
                all_detailed_metrics_rf_results.append({
                    'City': city_name_original, 'Scenario': scenario_name_safe, 'Repetition': rep,
                    'Seed': current_seed_loaded, 'Imputation_Method': RF_IMPUTER_METHOD_NAME,
                    'Variable': var_name, 'MAE': metrics_scores_rf[var_name]['MAE'], 
                    'RMSE': metrics_scores_rf[var_name]['RMSE'],
                    'NRMAE_range': metrics_scores_rf[var_name]['NRMAE_range'],
                    'NRMSE_range': metrics_scores_rf[var_name]['NRMSE_range']
                })
    print(f"  Procesamiento de escenarios RF para {city_name_original} completado.")

# --- Guardado de Métricas Resumidas (sin cambios en esta sección) ---
if all_detailed_metrics_rf_results:
    df_all_detailed_metrics_rf = pd.DataFrame(all_detailed_metrics_rf_results)
    print("\nGenerando resúmenes de métricas RF por variable y ciudad...")
    grouped_by_city_var_rf = df_all_detailed_metrics_rf.groupby(['City', 'Variable'])[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    grouped_by_city_var_rf.rename(columns={'MAE': 'Avg_MAE', 'RMSE': 'Avg_RMSE', 'NRMAE_range': 'Avg_NRMAE_range', 'NRMSE_range': 'Avg_NRMSE_range'}, inplace=True)
    for city_name_original_summary in grouped_by_city_var_rf['City'].unique(): # Renombrar variable de bucle
        city_name_safe_rf_summary = city_name_original_summary.replace(" ", "_").replace("-", "_")
        city_metrics_df_rf = grouped_by_city_var_rf[grouped_by_city_var_rf['City'] == city_name_original_summary]
        variables_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, f'metrics_variables_summary_{city_name_safe_rf_summary}.csv')
        city_metrics_df_rf.to_csv(variables_summary_path_rf, index=False)
        print(f"  Resumen RF de variables para {city_name_original_summary} guardado en: {variables_summary_path_rf}")
        if city_name_original_summary == df_air_quality_orig_load['City'].unique()[0]: print(city_metrics_df_rf)

    print("\nGenerando resumen RF combinado de contaminantes...")
    df_contaminants_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'].isin(CONTAMINANT_COLS)]
    contaminants_combined_summary_rf = df_contaminants_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    contaminants_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_Conts', 'RMSE': 'Avg_RMSE_Conts', 'NRMAE_range': 'Avg_NRMAE_range_Conts', 'NRMSE_range': 'Avg_NRMSE_range_Conts'}, inplace=True)
    contaminants_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_contaminants_combined_summary.csv')
    contaminants_combined_summary_rf.to_csv(contaminants_summary_path_rf, index=False)
    print(f"  Resumen RF combinado de contaminantes guardado en: {contaminants_summary_path_rf}")
    print(contaminants_combined_summary_rf)

    print("\nGenerando resumen RF combinado de AQI...")
    df_aqi_only_rf = df_all_detailed_metrics_rf[df_all_detailed_metrics_rf['Variable'] == 'AQI']
    aqi_combined_summary_rf = df_aqi_only_rf.groupby('City')[['MAE', 'RMSE', 'NRMAE_range', 'NRMSE_range']].mean().reset_index()
    aqi_combined_summary_rf.rename(columns={'MAE': 'Avg_MAE_AQI', 'RMSE': 'Avg_RMSE_AQI', 'NRMAE_range': 'Avg_NRMAE_range_AQI', 'NRMSE_range': 'Avg_NRMSE_range_AQI'}, inplace=True)
    aqi_summary_path_rf = os.path.join(METRICS_RF_OUTPUT_DIR, 'metrics_aqi_combined_summary.csv')
    aqi_combined_summary_rf.to_csv(aqi_summary_path_rf, index=False)
    print(f"  Resumen RF combinado de AQI guardado en: {aqi_summary_path_rf}")
    print(aqi_combined_summary_rf)
else:
    print("No se generaron resultados de métricas detalladas para Random Forest Imputer.")

script_end_time_rf = time.time()
print(f"\nProceso de Imputación con Random Forest completado.")
print(f"Archivos de datos simulados leídos desde: {SIMULATED_DATA_ROOT_DIR}")
print(f"Archivos de métricas resumidas RF guardados en: {METRICS_RF_OUTPUT_DIR}")
print(f"Archivos CSV imputados con RF guardados en subcarpetas dentro de: {IMPUTED_CSVS_RF_OUTPUT_DIR}")
print(f"Tiempo total de ejecución RF: {(script_end_time_rf - script_start_time_rf) / 60:.2f} minutos.")

Datos cargados desde: CSVs\Air_Quality.csv

Procesando Ciudad para Imputación RF: Brasilia
  Ajustando Scaler e IterativeImputer con RandomForest para Brasilia...
